# 🍛 FoodLens — Indian Food Classifier Training (Google Colab)

**Pehle ye karo (zaroori):** Runtime → Change runtime type → Hardware accelerator → **GPU (T4)** → Save.

Phir cells ko upar se neeche, ek-ek karke ▶️ (Shift+Enter) se chalao.

**Setup (ek baar karna hai):**
1. Google Drive mein ek folder banao, e.g. `MyDrive/FoodLens/`
2. Usme `food-classifier-code.zip` (project code) daal do
3. Dataset ke liye niche Cell 4 use karo — Hugging Face se seedha download ho jayega, Drive mein manually daalne ki zarurat nahi

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Google Drive mount karo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Project code unzip karo

`DRIVE_ZIP_PATH` ko apne actual Drive path se replace karo.

In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/FoodLens/food-classifier-code.zip"  # <-- apna path yaha daalo

!unzip -q "$DRIVE_ZIP_PATH" -d /content/
%cd /content/food-classifier-code
!ls

## 4. Indian Food dataset download karo (seedha Hugging Face se)

Ye cell dataset download karega, **Title Case folder names ko lowercase_with_underscore mein rename karega** (taaki `model/labels.txt` aur `nutrition_data.py` se exactly match ho), aur 70/15/15 train/val/test split bana dega.

In [ ]:
!pip install -q huggingface_hub

import os, re, shutil, random
from huggingface_hub import snapshot_download

RAW_DIR = snapshot_download(
    repo_id="SKSKCulinaryAI/Indian_Food_Images_Dataset",
    repo_type="dataset",
    local_dir="/content/raw_dataset",
)
SRC = os.path.join(RAW_DIR, "train")
print("Downloaded to:", SRC)
print("Classes found:", os.listdir(SRC))

In [ ]:
def normalize_name(name):
    """'Butter chicken' -> 'butter_chicken' — matches model/labels.txt"""
    name = name.strip().lower().replace(" ", "_")
    name = re.sub(r"[^a-z0-9_]", "", name)
    return name

DEST = "dataset"
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}
random.seed(42)

for split in SPLIT:
    os.makedirs(f"{DEST}/{split}", exist_ok=True)

classes = sorted(os.listdir(SRC))
for cls in classes:
    cls_dir = os.path.join(SRC, cls)
    if not os.path.isdir(cls_dir):
        continue
    norm_cls = normalize_name(cls)
    files = sorted([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    random.shuffle(files)
    n = len(files)
    n_train = int(n * SPLIT["train"])
    n_val = int(n * SPLIT["val"])
    buckets = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    for split, split_files in buckets.items():
        out_dir = f"{DEST}/{split}/{norm_cls}"
        os.makedirs(out_dir, exist_ok=True)
        for fname in split_files:
            shutil.copy(os.path.join(cls_dir, fname), os.path.join(out_dir, fname))
    print(f"{cls} -> {norm_cls}: {n} images ({len(buckets['train'])} train / {len(buckets['val'])} val / {len(buckets['test'])} test)")

print("\nSplit complete. Classes:", sorted(os.listdir(f'{DEST}/train')))

**Check:** the printed class list above should read exactly like `model/labels.txt` (25 classes: biryani, butter_chicken, chappati, ...). If a class is missing or spelled differently, `nutrition_data.py` won't have a match for it — open the file and add/rename the entry.

## 5. Dependencies install karo

In [ ]:
!pip install -q -r requirements.txt

## 6. Training chalao (2-phase: head warm-up → fine-tune)

`--backbone efficientnetb0` best accuracy ke liye hai. Sirf 25 classes aur ~10-30k images hone ki wajah se ye Food-101 se kaafi faster chalega — free Colab GPU pe roughly 20-40 minutes.

**Tip:** Pehli baar kam epochs se test karo (`--head-epochs 2 --finetune-epochs 3`) pipeline check karne ke liye, phir full run karo.

In [ ]:
!python train.py --dataset dataset --backbone efficientnetb0 \
    --head-epochs 8 --finetune-epochs 20 --batch-size 32

## 7. Trained model Drive mein save karo (zaroori — warna Colab session khatam hote hi delete ho jayega)

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/FoodLens/trained_model"  # <-- apna path yaha daalo
!mkdir -p "$SAVE_DIR"
!cp model/food_classifier.h5 "$SAVE_DIR/"
!cp model/labels.txt "$SAVE_DIR/"
print("Saved to Drive:", SAVE_DIR)

## 8. (Optional) Quick test — ek image pe prediction try karo

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image

model = load_model("model/food_classifier.h5")
with open("model/labels.txt") as f:
    labels = [l.strip() for l in f if l.strip()]

# apni koi test image ka path yahan daalo (dataset/test se koi file)
TEST_IMAGE = "dataset/test/biryani/REPLACE_WITH_REAL_FILENAME.jpg"  # <-- change this

img = Image.open(TEST_IMAGE).convert("RGB").resize((224, 224))
arr = preprocess_input(np.asarray(img, dtype="float32"))
arr = np.expand_dims(arr, axis=0)

probs = model.predict(arr)[0]
top_idx = np.argsort(probs)[::-1][:3]
for i in top_idx:
    print(f"{labels[i]}: {probs[i]*100:.2f}%")

## Agla step

Trained `food_classifier.h5` aur `labels.txt` ko Drive se download karke apne local project ke `model/` folder mein daal do. `app.py` mein `MODEL_BACKBONE = "efficientnetb0"` already set hai, to bas `python app.py` chalao — real predictions + nutrition + veg/non-veg badge sab live ho jayega.